In [1]:
from pathlib import Path
from maomao.hierarchical_structure.metadata import *
from maomao.hierarchical_structure.define_hierarchy_and_structure import *

### Building a Hierarchically Structured Peptide Toxicity Resource

This notebook integrates the processed peptide toxicity datasets into a unified, sequence-level resource covering multiple toxicity endpoints. It consolidates direct annotations, resolves endpoint-specific ambiguous evidence, and applies positive-only hierarchical relationships between toxicity effects.

Ambiguous annotations always take precedence over positive evidence inferred from a child endpoint. Therefore, hierarchical propagation can add a positive annotation only when the parent endpoint is not already classified as ambiguous.

The final pivot contains one row per unique peptide sequence and one encoded column per toxicity endpoint:

| Code | Meaning |
|---:|---|
| `0` | Negative |
| `1` | Positive |
| `2` | Ambiguous |
| `3` | Unlabeled |
| `999` | No information |

The notebook also creates a comprehensive, machine-readable metadata record that combines source-level provenance, endpoint-integration metadata, hierarchy rules, annotation statistics, ambiguity information, quality-control results, and output-file checksums.

- Auxiliary variables

In [2]:
PROCESSED_DATA_ROOT = Path("../../processed_data")

INPUT_ROOT = (PROCESSED_DATA_ROOT / "integrating_and_cleaning_data")
SOURCE_METADATA_ROOT = (PROCESSED_DATA_ROOT / "toxic_effect_classification")
OUTPUT_ROOT = (PROCESSED_DATA_ROOT / "processed_data")

INCLUDE_FULL_CROSS_PRODUCT = True

MIN_LENGTH = 5
MAX_LENGTH = 70
CANONICAL_RESIDUES = "ACDEFGHIKLMNPQRSTVWY"

RESOURCE_VERSION = "1.1.0"
REPOSITORY_URL = ("https://github.com/kren-ai-lab/maomao")
LICENSE_NAME = "See repository LICENSE.txt"

FINAL_METADATA_PATH = (OUTPUT_ROOT / "metadata.json")

- Review input files: This cell shows which files exist.

In [3]:
cfg = Config(
    input_root=INPUT_ROOT,
    output_root=OUTPUT_ROOT,
    min_length=MIN_LENGTH,
    max_length=MAX_LENGTH,
    canonical_residues=CANONICAL_RESIDUES,
    include_full_cross_product=(
        INCLUDE_FULL_CROSS_PRODUCT
    ),
)

input_manifest = discover_input_files(cfg)
display(input_manifest)

,endpoint,input_endpoint_folder,declared_status,filename,path,exists
0,toxic,toxic,positive,positive.csv,../../processed_data/integrating_and_cleaning_...,True
1,toxic,toxic,negative,negative.csv,../../processed_data/integrating_and_cleaning_...,True
2,toxic,toxic,ambiguous,ambiguous_data.csv,../../processed_data/integrating_and_cleaning_...,True
3,toxic,toxic,unlabeled,only_unlabel.csv,../../processed_data/integrating_and_cleaning_...,False
4,cytotoxic,cytotoxic,positive,positive.csv,../../processed_data/integrating_and_cleaning_...,True
5,cytotoxic,cytotoxic,negative,negative.csv,../../processed_data/integrating_and_cleaning_...,True
6,cytotoxic,cytotoxic,ambiguous,ambiguous_data.csv,../../processed_data/integrating_and_cleaning_...,True
7,cytotoxic,cytotoxic,unlabeled,only_unlabel.csv,../../processed_data/integrating_and_cleaning_...,False
8,hemolytic,hemolytic,positive,positive.csv,../../processed_data/integrating_and_cleaning_...,True
9,hemolytic,hemolytic,negative,negative.csv,../../processed_data/integrating_and_cleaning_...,True


- Build the pivot file

In [4]:
results = build_all(cfg)

In [5]:
print("Unique sequences:", results["metadata"]["n_unique_sequences"])
print("Results saved in:", OUTPUT_ROOT.resolve())

Unique sequences: 71857
Results saved in: /home/nicole/Descargas/maomao/processed_data/processed_data


- Summary and visualization of results

In [6]:
display(results["summary"]) # Summary data

status,toxicity_endpoint,positive,negative,ambiguous,unlabeled
0,anti_mammalian_cells,3664,17729,0,0
1,cytolysis,341,2,0,0
2,cytotoxic,7122,17238,1094,0
3,embryotoxic,2,0,0,0
4,hemolytic,4714,19350,6852,5044
5,ichthyotoxic,5,0,0,0
6,neurotoxic,1481,1700,2,0
7,toxic,13824,35487,7769,0


In [7]:
print("\nPivote file:")
print("Shape:", results["wide"].shape)
display(results["wide"].head())


Pivote file:
Shape: (71857, 10)


,id,sequence,toxic,cytotoxic,hemolytic,cytolysis,anti_mammalian_cells,neurotoxic,embryotoxic,ichthyotoxic
0,sha256_c70d828a62440a19acb4c85c0d1cff29a5b340a...,AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS,2,0,2,999,0,999,999,999
1,sha256_cb0e712239fad2596dfffa24ab580259aba066d...,AAAAAAAAAGETS,999,0,0,999,0,999,999,999
2,sha256_966434f9646239a8cd0a9375d1036fb4328d803...,AAAAAAAAAK,999,999,0,999,999,999,999,999
3,sha256_1de8b390ef5a86661bf0da4babf5a6059743fb6...,AAAAAAAIKMLMDLVNERIMALNKKAKK,0,0,0,999,0,999,999,999
4,sha256_42c3b829be38387546e22839e4d70387e2af55c...,AAAAARRRIRKQAHAHSK,0,0,0,999,0,999,999,999


- Working with metadata

In [8]:
resource_metadata = build_maomao_metadata(
    cfg=cfg,
    results=results,
    source_metadata_root=(
        SOURCE_METADATA_ROOT
    ),
    resource_version=RESOURCE_VERSION,
    repository_url=REPOSITORY_URL,
    license_name=LICENSE_NAME,
)

write_maomao_metadata(
    resource_metadata,
    FINAL_METADATA_PATH,
)

results["resource_metadata"] = (
    resource_metadata
)